In [1]:
import asyncio
import gc
import os
import signal
import subprocess
import sys
import time
from pathlib import Path

import requests
import torch
from huggingface_hub import snapshot_download
from sentence_transformers import SentenceTransformer

sys.path.append("src")
from run_bench import run_benchmark

/usr/local/lib/python3.10/dist-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[2026-03-01 14:24:35,268] [INFO] [real_accelerator.py:203:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [2]:
BASE_MODELS_DIR = Path("models")
BASE_MODELS_DIR.mkdir(parents=True, exist_ok=True)

VLLM_HOST = "127.0.0.1"
VLLM_PORT = 8000
VLLM_API = f"http://{VLLM_HOST}:{VLLM_PORT}/v1"
VLLM_KEY = "local-bench-key"

HF_TOKEN = os.getenv("HF_TOKEN", None)

BENCH_CONCURRENCY = 40

VLLM_GPU_MEMORY_UTIL = 0.90
VLLM_MAX_NUM_SEQS = 20
VLLM_MAX_MODEL_LEN = 8192

MODEL_SPECS = [
    ######{"alias": "Qwen3-8B", "repo_id": "Qwen/Qwen3-8B"},
    ######{"alias": "YandexGPT-5-Lite-8B-instruct", "repo_id": "yandex/YandexGPT-5-Lite-8B-instruct"},
    ######{"alias": "RuadaptQwen3-8B-Hybrid", "repo_id": "RefalMachine/RuadaptQwen3-8B-Hybrid"},
    
    # --- ВСЕ ЧТО ВЫШЕ - ПОСЧИТАНО ---
    
    #{"alias": "T-pro-it-2.1", "repo_id": "t-tech/T-pro-it-2.1"},
    #{"alias": "T-lite-it-2.1", "repo_id": "t-tech/T-lite-it-2.1"},
    #{"alias": "T-pro-it-2.0", "repo_id": "t-tech/T-pro-it-2.0"},
    #{"alias": "Qwen3-32B", "repo_id": "Qwen/Qwen3-32B"},
    #{"alias": "RuadaptQwen3-32B-Instruct", "repo_id": "RefalMachine/RuadaptQwen3-32B-Instruct"},
    #{"alias": "Qwen3-30B-A3B-Instruct-2507", "repo_id": "Qwen/Qwen3-30B-A3B-Instruct-2507"},
    #{"alias": "Qwen3-14B", "repo_id": "Qwen/Qwen3-14B"},
    
    #{"alias": "Qwen3-4B-Instruct", "repo_id": "Qwen/Qwen3-4B-Instruct-2507"},
    #{"alias": "RuadaptQwen3-4B-Instruct", "repo_id": "RefalMachine/RuadaptQwen3-4B-Instruct"},
    #{"alias": "Qwen3-4B", "repo_id": "Qwen/Qwen3-4B"},
    
    {"alias": "avibe", "repo_id": "AvitoTech/avibe"},
    {"alias": "GigaChat3-10B-A1.8B-bf16", "repo_id": "ai-sage/GigaChat3-10B-A1.8B-bf16"},
    {"alias": "T-lite-it-1.0", "repo_id": "t-tech/T-lite-it-1.0"},
    {"alias": "Vikhr-Nemo-12B-Instruct-R-21-09-24", "repo_id": "Vikhrmodels/Vikhr-Nemo-12B-Instruct-R-21-09-24"},
    {"alias": "RuadaptQwen2.5-7B-Lite-Beta", "repo_id": "RefalMachine/RuadaptQwen2.5-7B-Lite-Beta"},
]

In [3]:
def local_dir_for(alias: str) -> Path:
    safe = alias.replace("/", "_").replace(" ", "_")
    return BASE_MODELS_DIR / safe


def download_all_models():
    for spec in MODEL_SPECS:
        local_dir = local_dir_for(spec["alias"])
        spec["local_dir"] = str(local_dir)

        if local_dir.exists() and any(local_dir.iterdir()):
            print(f"[SKIP] Уже скачана: {spec['alias']} -> {local_dir}")
            continue

        print(f"[DOWNLOAD] {spec['alias']} <- {spec['repo_id']}")
        snapshot_download(
            repo_id=spec["repo_id"],
            local_dir=str(local_dir),
            token=HF_TOKEN,
        )
        print(f"[OK] {spec['alias']} -> {local_dir}")

In [5]:
download_all_models()

[SKIP] Уже скачана: T-pro-it-2.1 -> models/T-pro-it-2.1
[DOWNLOAD] Qwen3-32B <- Qwen/Qwen3-32B


Fetching 27 files:   0%|                                                                                                                                                                                                     | 0/27 [00:00<?, ?it/s]Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Fetching 27 files:   4%|███████                                                                                                                                                                                      | 1/27 [00:00<00:06,  4.22it/s]Xet Storage is enabled for this repo, but the 'h

[OK] Qwen3-32B -> models/Qwen3-32B
[DOWNLOAD] RuadaptQwen3-32B-Instruct <- RefalMachine/RuadaptQwen3-32B-Instruct


Fetching 25 files:   0%|                                                                                                                                                                                                     | 0/25 [00:00<?, ?it/s]Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' packa

[OK] RuadaptQwen3-32B-Instruct -> models/RuadaptQwen3-32B-Instruct
[DOWNLOAD] Qwen3-30B-A3B-Instruct-2507 <- Qwen/Qwen3-30B-A3B-Instruct-2507


...

[OK] T-lite-it-1.0 -> models/T-lite-it-1.0
[DOWNLOAD] Vikhr-Nemo-12B-Instruct-R-21-09-24 <- Vikhrmodels/Vikhr-Nemo-12B-Instruct-R-21-09-24


Fetching 17 files:   0%|                                                                                                                                                                                                     | 0/17 [00:00<?, ?it/s]Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Fetching 17 files:   6%|███████████                         

[OK] Vikhr-Nemo-12B-Instruct-R-21-09-24 -> models/Vikhr-Nemo-12B-Instruct-R-21-09-24
[DOWNLOAD] RuadaptQwen2.5-7B-Lite-Beta <- RefalMachine/RuadaptQwen2.5-7B-Lite-Beta


Fetching 15 files:   0%|                                                                                                                                                                                                     | 0/15 [00:00<?, ?it/s]Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' packa

[OK] RuadaptQwen2.5-7B-Lite-Beta -> models/RuadaptQwen2.5-7B-Lite-Beta


In [3]:
def wait_vllm_ready(timeout_sec: int = 1200):
    url = f"{VLLM_API}/models"
    headers = {"Authorization": f"Bearer {VLLM_KEY}"}
    deadline = time.time() + timeout_sec
    last_error = None

    while time.time() < deadline:
        try:
            r = requests.get(url, headers=headers, timeout=5)
            if r.ok:
                return
            last_error = f"{r.status_code}: {r.text[:200]}"
        except Exception as e:
            last_error = repr(e)

        time.sleep(2)

    raise RuntimeError(f"vLLM не поднялся. Последняя ошибка: {last_error}")


def start_vllm_server(spec: dict) -> subprocess.Popen:
    alias = spec["alias"]
    model_path = BASE_MODELS_DIR / alias.replace("/", "_").replace(" ", "_")

    cmd = [
        "vllm", "serve", model_path,
        "--host", VLLM_HOST,
        "--port", str(VLLM_PORT),
        "--api-key", VLLM_KEY,
        "--served-model-name", alias,
        "--tensor-parallel-size", "1",
        "--dtype", "auto",
        "--gpu-memory-utilization", str(VLLM_GPU_MEMORY_UTIL),
        "--max-model-len", str(VLLM_MAX_MODEL_LEN),
        "--max-num-seqs", str(VLLM_MAX_NUM_SEQS),
        "--swap-space", "16",
        "--generation-config", "vllm",
        "--disable-log-stats",
        "--disable-uvicorn-access-log",
        "--disable-log-requests"
    ]

    # cmd.append("--trust-remote-code")

    print(f"\n=== START vLLM: {alias} ===")
    env = os.environ.copy()
    env["VLLM_CONFIGURE_LOGGING"] = "0"
    
    # proc = subprocess.Popen(cmd, start_new_session=True)
    proc = subprocess.Popen(
        cmd,
        start_new_session=True,
        env=env,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.STDOUT,
    )
    wait_vllm_ready()
    print(f"[READY] {alias}")
    return proc


def stop_vllm_server(proc: subprocess.Popen):
    print("[STOP] vLLM")

    if proc and proc.poll() is None:
        try:
            os.killpg(proc.pid, signal.SIGTERM)
        except ProcessLookupError:
            pass

        try:
            proc.wait(timeout=30)
        except subprocess.TimeoutExpired:
            try:
                os.killpg(proc.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            proc.wait(timeout=10)

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass

    print("[CLEARED] CUDA cache")

In [4]:
async def run_benchmarks_for_model(model_name: str, encoder, encoder_device):
    common_kwargs = dict(
        api=VLLM_API,
        key=VLLM_KEY,
        model_name=model_name,
        concurrency=BENCH_CONCURRENCY,
        number_of_books=40,
        encoder_name="deepvk/USER-bge-m3",
        device="cpu",
        initial_word_limit=500,
        cap_chars=80000,
        shared_encoder=encoder,
        shared_device=encoder_device,
    )

    runs = [
        ("output_hierarchical_default", "hierarchical", "default"),
        ("output_hierarchical_filtered", "hierarchical", "filtered"),
        ("output_blueprint_default", "blueprint", "default"),
        ("output_blueprint_cluster", "blueprint", "cluster"),
    ]

    for output_dir, method, mode in runs:
        print(f"  -> {model_name}: {method}/{mode}")
        start = time.perf_counter()
        await run_benchmark(
            output_dir=output_dir,
            method=method,
            mode=mode,
            **common_kwargs,
        )
        end = time.perf_counter()
        print(f"Time for  -> {model_name}: {method}/{mode}: {end - start}")

In [5]:
# =========================
# ГЛАВНЫЙ ЦИКЛ
# =========================

async def main():

    encoder_device = torch.device("cuda")
    encoder = SentenceTransformer("deepvk/USER-bge-m3").to(encoder_device)

    # 3) vLLM -> bench -> no vLLM -> clear memo
    for spec in MODEL_SPECS:
        proc = None
        try:
            print(f"\n########## {spec['alias']} ##########")
            proc = start_vllm_server(spec)
            await run_benchmarks_for_model(spec["alias"], encoder, encoder_device)
        except Exception as e:
            print(f"[ERROR] {spec['alias']}: {e}")
        finally:
            if proc is not None:
                stop_vllm_server(proc)

    print("\nГотово.")

In [ ]:
await main()


########## avibe ##########

=== START vLLM: avibe ===
[READY] avibe
  -> avibe: hierarchical/default
Time for  -> avibe: hierarchical/default: 1267.4417764171958
  -> avibe: hierarchical/filtered


In [6]:
import os
import signal
import subprocess
import time
import requests

MODEL_PATH = "models/YandexGPT-5-Lite-8B-instruct"  # <-- замени на свою локальную папку
MODEL_NAME = "test-model"
HOST = "127.0.0.1"
PORT = 8000
API_KEY = "test-key"
BASE_URL = f"http://{HOST}:{PORT}/v1"


def gpu_mem_mb():
    """Суммарно занято GPU-памяти по всем GPU, в MB."""
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"]
        ).decode().strip().splitlines()
        return sum(int(x.strip()) for x in out if x.strip())
    except Exception:
        return None


def wait_ready(timeout=300):
    url = f"{BASE_URL}/models"
    headers = {"Authorization": f"Bearer {API_KEY}"}
    deadline = time.time() + timeout
    last_err = None

    while time.time() < deadline:
        try:
            r = requests.get(url, headers=headers, timeout=5)
            if r.ok:
                return
            last_err = f"{r.status_code}: {r.text[:200]}"
        except Exception as e:
            last_err = repr(e)
        time.sleep(2)

    raise RuntimeError(f"vLLM не поднялся: {last_err}")


def start_vllm():
    cmd = [
        "vllm", "serve", MODEL_PATH,
        "--host", HOST,
        "--port", str(PORT),
        "--api-key", API_KEY,
        "--served-model-name", MODEL_NAME,
        "--gpu-memory-utilization", "0.5",  # специально скромно
        "--max-model-len", "1024",
        "--max-num-seqs", "1",
    ]
    return subprocess.Popen(cmd, start_new_session=True)


def stop_vllm(proc):
    if proc.poll() is None:
        try:
            os.killpg(proc.pid, signal.SIGTERM)
        except ProcessLookupError:
            pass

        try:
            proc.wait(timeout=20)
        except subprocess.TimeoutExpired:
            try:
                os.killpg(proc.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            proc.wait(timeout=10)


# 1) Сколько памяти было до старта
mem_before = gpu_mem_mb()
print("GPU before:", mem_before, "MB")

# 2) Поднимаем сервер
proc = start_vllm()
wait_ready()
time.sleep(2)

mem_after_start = gpu_mem_mb()
print("GPU after start:", mem_after_start, "MB")

# 3) Делаем один тестовый запрос
headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
}
payload = {
    "model": MODEL_NAME,
    "messages": [
        {"role": "user", "content": "Напиши одно слово: привет"}
    ],
    "max_tokens": 8,
    "temperature": 0,
}

r = requests.post(f"{BASE_URL}/chat/completions", headers=headers, json=payload, timeout=120)
r.raise_for_status()
data = r.json()

print("Server response:")
print(data["choices"][0]["message"]["content"])

# 4) Гасим сервер
stop_vllm(proc)
time.sleep(3)

mem_after_stop = gpu_mem_mb()
print("GPU after stop:", mem_after_stop, "MB")

if mem_before is not None and mem_after_stop is not None:
    print("Freed:", mem_after_start - mem_after_stop, "MB")

GPU before: 7 MB


/usr/local/lib/python3.10/dist-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


INFO 03-01 07:54:30 [__init__.py:239] Automatically detected platform cuda.
INFO 03-01 07:54:31 [api_server.py:1034] vLLM API server version 0.8.4
INFO 03-01 07:54:31 [api_server.py:1035] args: Namespace(subparser='serve', model_tag='models/YandexGPT-5-Lite-8B-instruct', config='', host='127.0.0.1', port=8000, uvicorn_log_level='info', disable_uvicorn_access_log=False, allow_credentials=False, allowed_origins=['*'], allowed_methods=['*'], allowed_headers=['*'], api_key='test-key', lora_modules=None, prompt_adapters=None, chat_template=None, chat_template_content_format='auto', response_role='assistant', ssl_keyfile=None, ssl_certfile=None, ssl_ca_certs=None, enable_ssl_refresh=False, ssl_cert_reqs=0, root_path=None, middleware=[], return_tokens_as_token_ids=False, disable_frontend_multiprocessing=False, enable_request_id_headers=False, enable_auto_tool_choice=False, tool_call_parser=None, tool_parser_plugin='', model='models/YandexGPT-5-Lite-8B-instruct', task='auto', tokenizer=None, h

/usr/local/lib/python3.10/dist-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


INFO 03-01 07:54:50 [__init__.py:239] Automatically detected platform cuda.
INFO 03-01 07:54:53 [core.py:61] Initializing a V1 LLM engine (v0.8.4) with config: model='models/YandexGPT-5-Lite-8B-instruct', speculative_config=None, tokenizer='models/YandexGPT-5-Lite-8B-instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=1024, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='auto', reasoning_backend=None), observability_config=ObservabilityConfig(show_hidden_metrics=False, otlp_traces_endpoint=None, collect_model_forward_time=False, collect_model_execute_time=False), seed=None, served_model_name=test-model, num_scheduler_steps=1, multi_step_stream_outp

...

INFO 03-01 07:54:57 [loader.py:458] Loading weights took 2.19 seconds
INFO 03-01 07:54:57 [gpu_model_runner.py:1291] Model loading took 14.9771 GiB and 2.412691 seconds
INFO 03-01 07:55:04 [backends.py:416] Using cache directory: /root/.cache/vllm/torch_compile_cache/2de34b405f/rank_0_0 for vLLM's torch.compile
INFO 03-01 07:55:04 [backends.py:426] Dynamo bytecode transform time: 7.20 s
INFO 03-01 07:55:07 [backends.py:132] Cache the graph of shape None for later use
INFO 03-01 07:55:33 [backends.py:144] Compiling a graph for general shape takes 28.18 s
INFO 03-01 07:55:44 [monitor.py:33] torch.compile takes 35.38 s in total
INFO 03-01 07:55:45 [kv_cache_utils.py:634] GPU KV cache size: 193,200 tokens
INFO 03-01 07:55:45 [kv_cache_utils.py:637] Maximum concurrency for 1,024 tokens per request: 188.67x
INFO 03-01 07:56:13 [gpu_model_runner.py:1626] Graph capturing finished in 27 secs, took 0.51 GiB
INFO 03-01 07:56:13 [core.py:163] init engine (profile, create kv cache, warmup model) to

INFO:     Started server process [1512]
INFO:     Waiting for application startup.
INFO:     Application startup complete.


INFO:     127.0.0.1:46610 - "GET /v1/models HTTP/1.1" 200 OK
GPU after start: 40601 MB
INFO 03-01 07:56:21 [chat_utils.py:396] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.
INFO 03-01 07:56:21 [logger.py:39] Received request chatcmpl-c5e54ac651104b988c1c5312754152d4: prompt: '<s> Пользователь: Напиши одно слово: привет\n\n Ассистент:[SEP]', params: SamplingParams(n=1, presence_penalty=0.0, frequency_penalty=0.0, repetition_penalty=1.0, temperature=0.0, top_p=1.0, top_k=-1, min_p=0.0, seed=None, stop=[], stop_token_ids=[], bad_words=[], include_stop_str_in_output=False, ignore_eos=False, max_tokens=8, min_tokens=0, logprobs=None, prompt_logprobs=None, skip_special_tokens=True, spaces_between_special_tokens=True, truncate_prompt_tokens=None, guided_decoding=None, extra_args=None), prompt_token_ids: None, lora_request: None, prompt_adapter_request: None.
INFO 03-01 07:56:21 [async_llm.py:228] Added request chatcmpl

[rank0]:[W301 07:56:22.595190265 ProcessGroupNCCL.cpp:1496] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


INFO 03-01 07:56:23 [loggers.py:87] Engine 000: Avg prompt throughput: 0.1 tokens/s, Avg generation throughput: 0.0 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 0.0%
GPU after stop: 7 MB
Freed: 40594 MB
